# Reducing dimension: PCA and SVD

The slides computed PCA by SVD on scikit-learn's handwritten digits (`load_digits`: 1,797 8 × 8 grayscale images, from 0 to 16, that is 64 columns). This notebook practices it as a compression of the data, before the model:

1. The concentration of distances
2. PCA by SVD
3. Choosing k
4. PCA in a pipeline
5. Sparse data: `TruncatedSVD`
6. Data in batches: `IncrementalPCA`
7. Projecting new points: PCA, UMAP, t-SNE

The exercises are independent, from the simplest to the longest: the last ones are optional. No GPU needed.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz

import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.25, stratify=y, random_state=0
)
print(X.shape, X_train.shape, X_test.shape)


def show_digits(images: np.ndarray, title: str) -> None:
  # Shows 8 × 8 images side by side, one per row of `images`
  fig, axes = plt.subplots(1, len(images), figsize=(len(images), 1.5))
  for ax, image in zip(axes, images):
    ax.imshow(image.reshape(8, 8), cmap="gray_r")
    ax.axis("off")
  fig.suptitle(title)
  plt.show()


show_digits(X[:10], "The first 10 images")

## 1. The concentration of distances

In high dimension, a point's nearest and farthest neighbors are almost at the same distance (the slide "The Curse of Dimensionality"). Check it:

- write `distance_ratio(points, query)`, the ratio of the largest to the smallest Euclidean distance from `query` to the rows of `points`;
- for d = 2, 10, 100, 1,000 and 10,000, draw 1,000 uniform points in $[0, 1]^d$ and a query point, and print the median of that ratio over 20 draws;
- then on the digits: the median ratio for 20 images against the 1,796 others, and for 1,796 uniform points of dimension 64.

Do the digits behave like uniform points of the same dimension? What does that say about the number of directions they really need?

**Useful documentation**

- [`numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)
- [`numpy.random.Generator.random`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.random.html)
- [`numpy.delete`](https://numpy.org/doc/stable/reference/generated/numpy.delete.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `np.linalg.norm(points - query, axis=1)`: the distance from `query` to each row of `points` (numpy repeats `query` over each row)
- `rng = np.random.default_rng(0)`, then `rng.random((1_000, d))` for 1,000 points and `rng.random(d)` for the query
- `np.delete(X, i, axis=0)`: every image but the `i`-th; `rng.choice(len(X), 20, replace=False)` draws 20 distinct indices
- the expected order of magnitude: over 100 in dimension 2, about 1.03 in dimension 10,000

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def distance_ratio(points: np.ndarray, query: np.ndarray) -> float:
  distances = ...
  return ...


rng = np.random.default_rng(0)
for d in [2, 10, 100, 1_000, 10_000]:
  ratios = [distance_ratio(..., ...) for _ in range(20)]
  print(f"d = {d}: {np.median(ratios):.2f}")

indices = rng.choice(...)
ratios = [distance_ratio(np.delete(X, i, axis=0), X[i]) for i in indices]
print(f"Digits (d = 64): {np.median(ratios):.2f}")
ratios = [distance_ratio(..., ...) for _ in range(20)]
print(f"Uniform (d = 64): {np.median(ratios):.2f}")
```

### Solution

In [ ]:
def distance_ratio(points: np.ndarray, query: np.ndarray) -> float:
  distances = np.linalg.norm(points - query, axis=1)
  return distances.max() / distances.min()


rng = np.random.default_rng(0)
for d in [2, 10, 100, 1_000, 10_000]:
  ratios = [distance_ratio(rng.random((1_000, d)), rng.random(d)) for _ in range(20)]
  print(f"d = {d}: {np.median(ratios):.2f}")

indices = rng.choice(len(X), 20, replace=False)
ratios = [distance_ratio(np.delete(X, i, axis=0), X[i]) for i in indices]
print(f"Digits (d = 64): {np.median(ratios):.2f}")
ratios = [distance_ratio(rng.random((1_796, 64)), rng.random(64)) for _ in range(20)]
print(f"Uniform (d = 64): {np.median(ratios):.2f}")
# The digits keep a much larger ratio than uniform points of dimension 64: they only
# fill a small part of the space, around far fewer than 64 directions. That is what
# PCA exploits.

## 2. PCA by SVD

Compute the PCA of `X` as on the slide "PCA by SVD — Code":

- center `X`, decompose it with `np.linalg.svd`, then compute the k = 10 new columns `Z` and each axis's share of the variance;
- check that scikit-learn's `PCA(n_components=10)` gives the same columns, up to their sign, and the same shares of variance (`explained_variance_ratio_`);
- for k = 2, 5, 10 and 20, reconstruct `X` from its first k columns, show the first 10 reconstructed images with `show_digits`, and the mean squared error of the reconstruction.

From which k can you read the digits? How many values per image do you then store, instead of 64?

**Useful documentation**

- [`PCA.inverse_transform`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html#sklearn.decomposition.PCA.inverse_transform)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `X.mean(axis=0)`: each column's mean; `np.linalg.svd(Xc, full_matrices=False)` returns `U`, `S` (the singular values $\sigma_i$, already in decreasing order) and `Vt`
- the k new columns: `Xc @ Vt[:k].T`; the shares of variance: `S**2 / np.sum(S**2)`
- comparing up to the sign: `np.allclose(np.abs(a), np.abs(b))`, which must print `True`
- reconstructing: `Z @ Vt[:k] + mean`, with the k columns `Z` (or `pca.inverse_transform(Z)` with a fitted `PCA(k)`); the error: `np.mean((X - X_hat) ** 2)`
- with k = 64, the reconstruction is exact: the error is 0, up to rounding

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from sklearn.decomposition import PCA

mean = ...
Xc = X - mean
U, S, Vt = np.linalg.svd(...)
k = 10
Z = ...
ratio = ...

pca = PCA(n_components=k)
Z_sk = pca.fit_transform(X)
print(np.allclose(np.abs(Z_sk), np.abs(Z)))
print(np.allclose(..., ratio[:k]))

for k in [2, 5, 10, 20]:
  X_hat = ...
  error = np.mean((X - X_hat) ** 2)
  show_digits(X_hat[:10], f"k = {k}, mean squared error: {error:.2f}")
show_digits(X[:10], "Originals")
```

### Solution

In [ ]:
from sklearn.decomposition import PCA

mean = X.mean(axis=0)
Xc = X - mean
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
k = 10
Z = Xc @ Vt[:k].T  # = U[:, :k] * S[:k]
ratio = S**2 / np.sum(S**2)

pca = PCA(n_components=k)
Z_sk = pca.fit_transform(X)
print(np.allclose(np.abs(Z_sk), np.abs(Z)))
print(np.allclose(pca.explained_variance_ratio_, ratio[:k]))

for k in [2, 5, 10, 20]:
  X_hat = (Xc @ Vt[:k].T) @ Vt[:k] + mean
  error = np.mean((X - X_hat) ** 2)
  show_digits(X_hat[:10], f"k = {k}, mean squared error: {error:.2f}")
show_digits(X[:10], "Originals")
# The digits can be read from about k = 10 to 20: 10 to 20 values per image instead of
# 64, plus the k axes and the mean, stored once for the whole dataset.

## 3. Choosing k

- Fit a `PCA()` that keeps every axis, and plot the cumulative explained variance against k.
- Find the smallest k that keeps 80%, 95% and 99% of the variance, and check that `PCA(0.8)`, `PCA(0.95)` and `PCA(0.99)` find the same ones (their `n_components_` attribute).

What share of the 64 columns must you keep for 95% of the variance? Why does the curve rise so fast at first?

**Useful documentation**

- [`numpy.cumsum`](https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html)
- [`numpy.searchsorted`](https://numpy.org/doc/stable/reference/generated/numpy.searchsorted.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `np.cumsum(pca.explained_variance_ratio_)`: the variance kept by the first k axes, for k = 1 to 64
- `np.searchsorted(cumulative, 0.95) + 1`: the smallest k whose cumulative variance reaches 95% (`searchsorted` returns an index, which starts at 0)
- `plt.axhline(0.95, color="gray", linestyle="--")` marks the threshold on the plot
- the expected result: k = 29 for 95%

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from sklearn.decomposition import PCA

cumulative = np.cumsum(...)
plt.plot(np.arange(1, len(cumulative) + 1), cumulative, marker=".")
plt.axhline(0.95, color="gray", linestyle="--")
plt.xlabel("k")
plt.ylabel("Cumulative explained variance")
plt.show()

for share in [0.8, 0.95, 0.99]:
  k = ...
  n_components = ...
  print(f"{share:.0%}: k = {k} ({k / 64:.0%} of the columns), PCA: {n_components}")
```

### Solution

In [ ]:
from sklearn.decomposition import PCA

cumulative = np.cumsum(PCA().fit(X).explained_variance_ratio_)
plt.plot(np.arange(1, len(cumulative) + 1), cumulative, marker=".")
plt.axhline(0.95, color="gray", linestyle="--")
plt.xlabel("k")
plt.ylabel("Cumulative explained variance")
plt.show()

for share in [0.8, 0.95, 0.99]:
  k = np.searchsorted(cumulative, share) + 1
  n_components = PCA(share).fit(X).n_components_
  print(f"{share:.0%}: k = {k} ({k / 64:.0%} of the columns), PCA: {n_components}")
# 95% of the variance with 29 axes out of 64, less than half. The axes are sorted by
# decreasing variance, and a handful of directions carry most of the variations
# between the images.

## 4. PCA in a pipeline

The starting cell split `X` into `X_train` (75%) and `X_test` (25%). For k = 5, 10, 20, 30, then without PCA:

- fit a `Pipeline` (`StandardScaler`, `PCA(k)`, `KNeighborsClassifier`) on `X_train`;
- measure the fitting time, the time to predict `X_test`, and the accuracy on `X_test`;
- put the results in a pandas `DataFrame`, one row per k.

Which k keeps the accuracy obtained without PCA? On 64 columns, does PCA save time? Why fit it in the pipeline rather than once and for all on the whole `X`?

**Useful documentation**

- [`sklearn.pipeline.Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [`sklearn.neighbors.KNeighborsClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
- [`sklearn.metrics.accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
- [`time.perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `Pipeline([("scaler", StandardScaler()), ("pca", PCA(k)), ("model", KNeighborsClassifier())])`; without PCA, the same list without the `"pca"` step
- `pipeline.fit(X_train, y_train)`, then `pipeline.predict(X_test)` and `accuracy_score(y_test, predictions)`
- `start = time.perf_counter()` before, `time.perf_counter() - start` after: a duration in seconds
- `pd.DataFrame(rows).T`, with `rows[k] = {"accuracy": ..., ...}`: one row per k
- the times are about a millisecond, and vary from one run to the next: run the cell again before concluding
- the expected result: an accuracy of about 0.97 without PCA

A skeleton to copy into the exercise cell, replacing each `...`:

```python
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rows = {}
for k in [5, 10, 20, 30, None]:
  steps = [("scaler", StandardScaler())]
  if k is not None:
    steps.append(...)
  steps.append(("model", KNeighborsClassifier()))
  pipeline = Pipeline(steps)
  start = time.perf_counter()
  pipeline.fit(...)
  fit_time = time.perf_counter() - start
  start = time.perf_counter()
  predictions = ...
  predict_time = time.perf_counter() - start
  rows[k or "no PCA"] = {
    "accuracy": accuracy_score(...),
    "fit (ms)": 1000 * fit_time,
    "predict (ms)": 1000 * predict_time,
  }
pd.DataFrame(rows).T.round(3)
```

### Solution

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rows = {}
for k in [5, 10, 20, 30, None]:
  steps = [("scaler", StandardScaler())]
  if k is not None:
    steps.append(("pca", PCA(k)))
  steps.append(("model", KNeighborsClassifier()))
  pipeline = Pipeline(steps)
  start = time.perf_counter()
  pipeline.fit(X_train, y_train)
  fit_time = time.perf_counter() - start
  start = time.perf_counter()
  predictions = pipeline.predict(X_test)
  predict_time = time.perf_counter() - start
  rows[k or "no PCA"] = {
    "accuracy": accuracy_score(y_test, predictions),
    "fit (ms)": 1000 * fit_time,
    "predict (ms)": 1000 * predict_time,
  }
# From about k = 20 to 30, the accuracy matches the one without PCA. On 64 columns,
# PCA hardly saves any time: its own computation costs as much as, or more than, what
# it saves the KNN. It pays off in high dimension, and that is measured on the target
# hardware. In the pipeline, PCA only learns from X_train, like the StandardScaler:
# fitted on the whole X, it would have seen the test set, and the evaluation would be
# optimistic.
pd.DataFrame(rows).T.round(3)

## 5. Sparse data: `TruncatedSVD`

The next cell reads 11,314 messages from 20 Usenet forums (the 20 Newsgroups dataset, which the starting cell downloaded) and represents them as TF-IDF: `X_text`, a sparse matrix, with one column per word. Then:

- how many columns, and what share of the values is nonzero? How many bytes does the matrix take as a sparse matrix (its three arrays `data`, `indices` and `indptr`), and how many would it take as a dense one, at 8 bytes per value?
- reduce it to 100 columns with `TruncatedSVD`, and measure the computation time and the share of the variance kept;
- do the same with `PCA`, which centers the data without building the centered matrix (see the hints for its solver).

Why not center `X_text` yourself, with `X_text - X_text.mean(axis=0)` (don't run it: it would take gigabytes of memory)? What does the share of variance kept by 100 axes say, compared with the digits?

**Useful documentation**

- [`scipy.sparse.csr_matrix`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.html)
- [`sklearn.decomposition.TruncatedSVD`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)

In [ ]:
from sklearn.datasets import load_files
from sklearn.feature_extraction.text import TfidfVectorizer

news = load_files("20news-bydate-train", encoding="latin-1")
X_text = TfidfVectorizer(min_df=2).fit_transform(news.data)
print(type(X_text).__name__, X_text.shape)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `X_text.shape`; `X_text.nnz`: the number of nonzero values; `X_text.data.nbytes + X_text.indices.nbytes + X_text.indptr.nbytes`: the sparse size
- `TruncatedSVD(n_components=100, random_state=0)`, then `fit(X_text)`; `svd.explained_variance_ratio_.sum()`: the share of variance kept
- `PCA(n_components=100, svd_solver="arpack", random_state=0)`: on a sparse matrix, the `"arpack"` solver centers implicitly, without densifying
- count one to two minutes for both

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from sklearn.decomposition import PCA, TruncatedSVD

n, d = X_text.shape
sparse_bytes = ...
print(f"{d} columns, {X_text.nnz / (n * d):.2%} nonzero values")
print(f"Sparse: {sparse_bytes / 1e6:.0f} MB, dense: {... / 1e9:.1f} GB")

for model in [TruncatedSVD(...), PCA(...)]:
  start = time.perf_counter()
  model.fit(X_text)
  elapsed = time.perf_counter() - start
  kept = ...
  print(f"{type(model).__name__}: {elapsed:.1f} s, variance kept {kept:.0%}")
```

### Solution

In [ ]:
from sklearn.decomposition import PCA, TruncatedSVD

n, d = X_text.shape
sparse_bytes = X_text.data.nbytes + X_text.indices.nbytes + X_text.indptr.nbytes
print(f"{d} columns, {X_text.nnz / (n * d):.2%} nonzero values")
print(f"Sparse: {sparse_bytes / 1e6:.0f} MB, dense: {8 * n * d / 1e9:.1f} GB")

for model in [
  TruncatedSVD(n_components=100, random_state=0),
  PCA(n_components=100, svd_solver="arpack", random_state=0),
]:
  start = time.perf_counter()
  model.fit(X_text)
  elapsed = time.perf_counter() - start
  kept = model.explained_variance_ratio_.sum()
  print(f"{type(model).__name__}: {elapsed:.1f} s, variance kept {kept:.0%}")
# About 56,000 columns and 0.3% nonzero values: about twenty MB as a sparse matrix,
# 5 GB as a dense one. Centering it yourself would make the matrix dense: a column
# minus its (nonzero) mean has no zeros left. TruncatedSVD doesn't center; PCA with
# "arpack" centers implicitly, more slowly, and keeps about the same variance. 100 axes
# only keep about 13% of the text's variance, when 29 keep 95% of the digits': the
# vocabulary varies in many more directions. On text, 100 to 300 axes are often kept
# (latent semantic analysis).

## 6. Data in batches: `IncrementalPCA`

When the data doesn't fit in memory, `IncrementalPCA` fits PCA batch by batch. Simulate it on the digits:

- split `X` into 10 batches, and fit an `IncrementalPCA` with 10 components with `partial_fit`, batch after batch;
- compare its shares of variance and its axes (`components_`) with those of a 10-component `PCA` fitted on the whole `X` at once.

Are the axes the same? Which ones differ the most, and why?

**Useful documentation**

- [`IncrementalPCA.partial_fit`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.IncrementalPCA.html#sklearn.decomposition.IncrementalPCA.partial_fit)
- [`numpy.array_split`](https://numpy.org/doc/stable/reference/generated/numpy.array_split.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `np.array_split(X, 10)`: a list of 10 batches of about 180 rows; each batch must have at least as many rows as components
- `ipca = IncrementalPCA(n_components=10)`, then `ipca.partial_fit(batch)` for each batch
- the axes have norm 1, and are defined up to their sign: `np.abs(np.sum(ipca.components_ * pca.components_, axis=1))`, the cosine between the axes of the same rank, is 1 for two identical axes

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from sklearn.decomposition import PCA, IncrementalPCA

ipca = IncrementalPCA(n_components=10)
for batch in ...:
  ...
pca = PCA(n_components=10).fit(X)
print("Variance, IncrementalPCA:", ipca.explained_variance_ratio_.round(3))
print("Variance, PCA:", ...)
print("Cosine between the axes:", ...)
```

### Solution

In [ ]:
from sklearn.decomposition import PCA, IncrementalPCA

ipca = IncrementalPCA(n_components=10)
for batch in np.array_split(X, 10):
  ipca.partial_fit(batch)
pca = PCA(n_components=10).fit(X)
print("Variance, IncrementalPCA:", ipca.explained_variance_ratio_.round(3))
print("Variance, PCA:", pca.explained_variance_ratio_.round(3))
cosines = np.abs(np.sum(ipca.components_ * pca.components_, axis=1))
print("Cosine between the axes:", cosines.round(3))
# The shares of variance and the first axes match (cosine of 1.000). The last ones
# drift a little (cosine around 0.98): their variances are close, and fitting batch by
# batch only tells them apart approximately.

## 7. Projecting new points: PCA, UMAP, t-SNE

In production, dimensionality reduction must transform data it didn't see when it was fitted. Compare three projections in 2 dimensions:

- fit a `PCA` and a `umap.UMAP` on `X_train`, then transform `X_train` and `X_test`;
- for each: the fitting time, the accuracy of a `KNeighborsClassifier` fitted on the projected `X_train` and evaluated on the projected `X_test`, and the scatter plot of the test points, colored by digit;
- does `sklearn.manifold.TSNE` have a `transform` method? Plot next to them the scatter plot its `fit_transform` gives on the whole `X`.

Which one separates the digits best? Which ones can serve in a production pipeline?

**Useful documentation**

- [`umap.UMAP`](https://umap-learn.readthedocs.io/en/latest/api.html)
- [`sklearn.manifold.TSNE`](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)
- [`sklearn.neighbors.KNeighborsClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
- [`matplotlib.axes.Axes.scatter`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.scatter.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `PCA(n_components=2)` and `umap.UMAP(n_components=2, random_state=0, n_jobs=1)`: `fit(X_train)`, then `transform(X_train)` and `transform(X_test)`; `random_state` fixes the result and imposes a single core, which `n_jobs=1` states without a warning
- `KNeighborsClassifier().fit(Z_train, y_train).score(Z_test, y_test)`: the accuracy on the projected test set
- `hasattr(TSNE(), "transform")` says whether the method exists; `TSNE(n_components=2, random_state=0).fit_transform(X)` places all the points at once
- `ax.scatter(Z[:, 0], Z[:, 1], c=labels, cmap="tab10", s=5)` colors each point by its digit
- count a minute or two: UMAP compiles its code on its first call, and t-SNE is slow

A skeleton to copy into the exercise cell, replacing each `...`:

```python
import umap
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier

print("TSNE has a transform method:", hasattr(TSNE(), "transform"))

_, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, reducer in zip(axes, [PCA(n_components=2), umap.UMAP(...)]):
  start = time.perf_counter()
  reducer.fit(...)
  fit_time = time.perf_counter() - start
  Z_train, Z_test = ..., ...
  accuracy = ...
  name = type(reducer).__name__
  print(f"{name}: fit {fit_time:.1f} s, accuracy {accuracy:.2f}")
  ax.scatter(Z_test[:, 0], Z_test[:, 1], c=y_test, cmap="tab10", s=5)
  ax.set_title(f"{name} (test)")
Z = ...
axes[2].scatter(..., c=y, cmap="tab10", s=5)
axes[2].set_title("t-SNE (whole X)")
plt.show()
```

### Solution

In [ ]:
import umap
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier

print("TSNE has a transform method:", hasattr(TSNE(), "transform"))

_, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, reducer in zip(
  axes, [PCA(n_components=2), umap.UMAP(n_components=2, random_state=0, n_jobs=1)]
):
  start = time.perf_counter()
  reducer.fit(X_train)
  fit_time = time.perf_counter() - start
  Z_train, Z_test = reducer.transform(X_train), reducer.transform(X_test)
  accuracy = KNeighborsClassifier().fit(Z_train, y_train).score(Z_test, y_test)
  name = type(reducer).__name__
  print(f"{name}: fit {fit_time:.1f} s, accuracy {accuracy:.2f}")
  ax.scatter(Z_test[:, 0], Z_test[:, 1], c=y_test, cmap="tab10", s=5)
  ax.set_title(f"{name} (test)")
Z = TSNE(n_components=2, random_state=0).fit_transform(X)
axes[2].scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=5)
axes[2].set_title("t-SNE (whole X)")
plt.show()
# In 2 dimensions, PCA mixes the digits up (accuracy around 0.6); UMAP separates them
# almost as well as the 64 columns do (around 0.97). t-SNE has no transform: it only
# places the points it was fitted on, and a new point means doing it all again. PCA and
# UMAP can serve in a pipeline; t-SNE is for visualizing.